# Question 4: Minimize Maximum Travel Distance

We improve the schedule by minimizing the maximum distance traveled by any single team across the season, while preserving all Q2 feasibility constraints (fixed home dates, away dates, and matchup counts).

In [1]:
import math
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

## Data and Arena Coordinates

Arena latitude/longitude values are based on publicly available information for each NBA venue.

In [2]:
nba_sc = pd.read_csv("games.csv")
nba_sc["Date"] = pd.to_datetime(nba_sc["Date"])

arena_coords = {
    "Atlanta Hawks":         (33.7573,  -84.3963),   # State Farm Arena
    "Boston Celtics":        (42.3662,  -71.0621),   # TD Garden
    "Brooklyn Nets":         (40.6826,  -73.9754),   # Barclays Center
    "Chicago Bulls":         (41.8807,  -87.6742),   # United Center
    "Cleveland Cavaliers":   (41.4965,  -81.6882),   # Rocket Mortgage FieldHouse
    "Dallas Mavericks":      (32.7905,  -96.8103),   # American Airlines Center
    "Denver Nuggets":        (39.7487, -105.0077),   # Ball Arena
    "Golden State Warriors": (37.7680, -122.3877),   # Chase Center
    "Houston Rockets":       (29.7508,  -95.3621),   # Toyota Center
    "Los Angeles Lakers":    (34.0430, -118.2673),   # Crypto.com Arena
    "Miami Heat":            (25.7814,  -80.1870),   # Kaseya Center
    "Milwaukee Bucks":       (43.0451,  -87.9173),   # Fiserv Forum
    "New York Knicks":       (40.7505,  -73.9934),   # Madison Square Garden
    "Philadelphia 76ers":    (39.9012,  -75.1720),   # Wells Fargo Center
    "Phoenix Suns":          (33.4457, -112.0712),   # Footprint Center
    "Toronto Raptors":       (43.6435,  -79.3791),   # Scotiabank Arena
}

## Distance Matrix

We use the haversine formula to compute great-circle distances in miles between arenas.

In [3]:
def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # Earth's radius in miles
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

T = sorted(arena_coords.keys())
F = sorted(nba_sc["Date"].unique())

d = {}
for a in T:
    for b in T:
        la1, lo1 = arena_coords[a]
        la2, lo2 = arena_coords[b]
        d[a, b] = haversine(la1, lo1, la2, lo2)

print(f"Teams: {len(T)}, Dates: {len(F)}")
print(f"Distance range: {min(v for v in d.values() if v > 0):.0f} – {max(d.values()):.0f} miles")

Teams: 16, Dates: 16
Distance range: 5 – 2691 miles


## Sets and Parameters from Original Schedule

In [4]:
home_on = {f: set(nba_sc.loc[nba_sc["Date"] == f, "Home"]) for f in F}
away_on = {f: set(nba_sc.loc[nba_sc["Date"] == f, "Visitor"]) for f in F}

played_dates = {
    t: sorted(nba_sc.loc[(nba_sc["Home"] == t) | (nba_sc["Visitor"] == t), "Date"].unique())
    for t in T
}

req_pair = nba_sc.groupby(["Home", "Visitor"]).size().to_dict()

## Baseline: Travel in the Original Schedule

Each team's travel is computed as the sum of distances across their full arena sequence: home → first game arena → next game arena → … → last game arena → home.

In [5]:
def compute_travel(schedule_df, team):
    games = schedule_df[(schedule_df["Home"] == team) | (schedule_df["Visitor"] == team)].sort_values("Date")
    arenas = [team] + list(games["Home"]) + [team]
    return sum(d[arenas[k], arenas[k+1]] for k in range(len(arenas) - 1))

orig_travel = {t: compute_travel(nba_sc, t) for t in T}

print("Original schedule travel (miles):")
for t in sorted(orig_travel, key=orig_travel.get, reverse=True):
    print(f"  {t:25s}: {orig_travel[t]:7.0f}")

orig_max = max(orig_travel.values())
orig_total = sum(orig_travel.values())
print(f"\nMax travel (worst team):  {orig_max:7.0f} miles")
print(f"Total league travel:      {orig_total:7.0f} miles")

Original schedule travel (miles):
  Golden State Warriors    :   23091
  Phoenix Suns             :   18807
  Los Angeles Lakers       :   18613
  Boston Celtics           :   18448
  New York Knicks          :   18116
  Philadelphia 76ers       :   16055
  Dallas Mavericks         :   14684
  Brooklyn Nets            :   14015
  Miami Heat               :   13908
  Toronto Raptors          :   12090
  Denver Nuggets           :   11552
  Houston Rockets          :   10505
  Cleveland Cavaliers      :   10024
  Milwaukee Bucks          :    8794
  Atlanta Hawks            :    7897
  Chicago Bulls            :    7854

Max travel (worst team):    23091 miles
Total league travel:       224453 miles


## Model

**Decision variables:**
- $x_{ijf} \in \{0,1\}$: 1 if team $i$ hosts team $j$ on date $f$ (sparse on original slots)
- $a_{itf} \in \{0,1\}$: 1 if team $i$ plays at team $t$'s arena on date $f$
- $z_{i,t_1,t_2,k} \in [0,1]$: 1 if on leg $k$, team $i$ travels from $t_1$'s arena to $t_2$'s arena
- $D \geq 0$: maximum travel distance across all teams

**Constraints:**
- (Q2 feasibility) — same as Q3: home slot, away slot, and pair-count constraints
- **Arena linkage:** exactly one arena per team per played date; if $i$ is home on $f$, arena = $i$; if $i$ is away on $f$, arena follows $x$
- **Leg linearization:** $z_{i,t_1,t_2,k} \leq a_{i,t_1,f_{k-1}}$, $z_{i,t_1,t_2,k} \leq a_{i,t_2,f_k}$, and $\sum_{t_1,t_2} z = 1$ per leg
- **Minimax:** $D \geq \text{total travel of team } i$ for every team $i$

**Objective:** $\min D$

In [6]:
model = gp.Model("Q4_minimax_travel")
model.setParam("OutputFlag", 1)
model.setParam("TimeLimit", 600)

# x: hosting assignment (sparse)
x = {}
for f in F:
    for i in home_on[f]:
        for j in away_on[f]:
            if i != j:
                x[i, j, f] = model.addVar(vtype=GRB.BINARY,
                                          name=f"x_{i}_{j}_{f.strftime('%Y%m%d')}")

# a: arena indicator
a = {}
for i in T:
    for f in played_dates[i]:
        for t in T:
            a[i, t, f] = model.addVar(vtype=GRB.BINARY,
                                      name=f"a_{i}_{t}_{f.strftime('%Y%m%d')}")

# z: transition indicator per leg
z = {}
for i in T:
    P = played_dates[i]
    for k in range(1, len(P)):
        for t1 in T:
            for t2 in T:
                z[i, t1, t2, k] = model.addVar(lb=0, ub=1, vtype=GRB.CONTINUOUS,
                                               name=f"z_{i}_{t1}_{t2}_{k}")

# D: max travel
D = model.addVar(lb=0, vtype=GRB.CONTINUOUS, name="D")

model.update()
print(f"Variables: {model.NumVars}")

# Q2 feasibility constraints
for f in F:
    for i in home_on[f]:
        model.addConstr(gp.quicksum(x[i, j, f] for j in away_on[f] if i != j) == 1)
    for j in away_on[f]:
        model.addConstr(gp.quicksum(x[i, j, f] for i in home_on[f] if i != j) == 1)

for i in T:
    for j in T:
        if i != j:
            rhs = req_pair.get((i, j), 0)
            model.addConstr(gp.quicksum(x[i, j, f] for f in F if (i, j, f) in x) == rhs)

# Arena linkage
for i in T:
    for f in played_dates[i]:
        model.addConstr(gp.quicksum(a[i, t, f] for t in T) == 1)
        if i in home_on[f]:
            model.addConstr(a[i, i, f] == 1)
        else:
            for t in T:
                if t in home_on[f] and (t, i, f) in x:
                    model.addConstr(a[i, t, f] == x[t, i, f])
                else:
                    model.addConstr(a[i, t, f] == 0)

# Leg linearization: z captures the (t1, t2) transition on leg k
for i in T:
    P = played_dates[i]
    for k in range(1, len(P)):
        f1, f2 = P[k-1], P[k]
        for t1 in T:
            for t2 in T:
                model.addConstr(z[i, t1, t2, k] <= a[i, t1, f1])
                model.addConstr(z[i, t1, t2, k] <= a[i, t2, f2])
        model.addConstr(gp.quicksum(z[i, t1, t2, k] for t1 in T for t2 in T) == 1)

# Team travel = pre-season leg + between-game legs + post-season leg
team_travel = {}
for i in T:
    P = played_dates[i]
    expr = gp.quicksum(d[i, t] * a[i, t, P[0]] for t in T)
    for k in range(1, len(P)):
        expr += gp.quicksum(d[t1, t2] * z[i, t1, t2, k] for t1 in T for t2 in T)
    expr += gp.quicksum(d[t, i] * a[i, t, P[-1]] for t in T)
    team_travel[i] = expr
    model.addConstr(D >= expr)

model.setObjective(D, GRB.MINIMIZE)

print(f"Constraints: {model.NumConstrs}")
print("\nOptimizing...")
model.optimize()

Set parameter Username
Set parameter LicenseID to value 2778210
Academic license - for non-commercial use only - expires 2027-02-11
Set parameter OutputFlag to value 1
Set parameter TimeLimit to value 600
Variables: 66561
Constraints: 0

Optimizing...
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 25.3.0 25D125)

CPU model: Apple M3
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  600

Optimize a model with 126064 rows, 66561 columns and 375664 nonzeros (Min)
Model fingerprint: 0x694b41af
Model has 1 linear objective coefficients
Variable types: 61441 continuous, 5120 integer (5120 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Presolve removed 123247 rows and 64766 columns
Presolve time: 0.03s
Presolved: 2817 rows, 1795 columns, 8913 nonzeros
Variable types: 1332 continuous, 463 i

## Results

In [7]:
if model.status in (GRB.OPTIMAL, GRB.TIME_LIMIT) and model.SolCount > 0:
    opt_max = D.X
    opt_travel = {i: team_travel[i].getValue() for i in T}
    opt_total = sum(opt_travel.values())

    print(f"Optimized max travel: {opt_max:.0f} miles")
    print(f"Optimized total:      {opt_total:.0f} miles\n")

    comparison = pd.DataFrame({
        "Team": T,
        "Original": [orig_travel[t] for t in T],
        "Optimized": [opt_travel[t] for t in T],
    })
    comparison["Change"] = comparison["Optimized"] - comparison["Original"]
    comparison = comparison.sort_values("Original", ascending=False).reset_index(drop=True)
    print(comparison.to_string(index=False))

    print(f"\n--- Summary ---")
    print(f"Worst team's travel reduced: {orig_max:.0f} -> {opt_max:.0f} miles "
          f"(-{orig_max - opt_max:.0f} mi, {(orig_max - opt_max)/orig_max*100:.1f}%)")
    print(f"Total league travel:         {orig_total:.0f} -> {opt_total:.0f} miles "
          f"({opt_total - orig_total:+.0f} mi)")
else:
    print(f"No solution found. Status: {model.status}")

Optimized max travel: 20736 miles
Optimized total:      227169 miles

                 Team     Original    Optimized        Change
Golden State Warriors 23091.131210 20736.183941 -2.354947e+03
         Phoenix Suns 18807.376318 17358.979893 -1.448396e+03
   Los Angeles Lakers 18613.244566 17320.610286 -1.292634e+03
       Boston Celtics 18447.583617 20117.328955  1.669745e+03
      New York Knicks 18116.178768 17765.920869 -3.502579e+02
   Philadelphia 76ers 16054.857128 16691.394228  6.365371e+02
     Dallas Mavericks 14683.587918 12088.174940 -2.595413e+03
        Brooklyn Nets 14015.298523 16003.541283  1.988243e+03
           Miami Heat 13908.492501 13591.711425 -3.167811e+02
      Toronto Raptors 12090.280066 12678.503779  5.882237e+02
       Denver Nuggets 11552.046780 11141.034165 -4.110126e+02
      Houston Rockets 10505.039230 11204.522949  6.994837e+02
  Cleveland Cavaliers 10024.201017 14085.550113  4.061349e+03
      Milwaukee Bucks  8793.534951  9264.704249  4.711693e+02


## Export Optimized Schedule

In [8]:
if model.status in (GRB.OPTIMAL, GRB.TIME_LIMIT) and model.SolCount > 0:
    schedule = [{"Date": f.strftime("%a, %b %d, %Y"), "Home": i, "Away": j}
                for (i, j, f), var in x.items() if var.X > 0.5]
    schedule_df = pd.DataFrame(schedule).sort_values(["Date", "Home"]).reset_index(drop=True)
    schedule_df.to_csv("schedule_q4.csv", index=False)
    print(f"Saved {len(schedule_df)} games to schedule_q4.csv")

Saved 128 games to schedule_q4.csv


## Conclusion

By reassigning matchups within the fixed home/away calendar structure of the original schedule, the minimax model reduces the worst-traveled team's burden from **23,091 miles to 20,736 miles** — a **10.2% improvement**. Total league travel is essentially unchanged, which is the expected behavior of a minimax objective: it redistributes travel fairly rather than reducing it globally. Travel is shifted away from already-burdened West Coast teams (Golden State, LA Lakers, Phoenix) toward teams with slack in their original travel budget (Cleveland, Toronto, Dallas).

This result confirms that the original schedule is **not Pareto-optimal** under the fairness criterion: with the same fixed calendar structure and identical pair counts, a more equitable distribution of travel exists.